# LeetCode 1711 — Count Good Meals bằng Python

Link đề: https://leetcode.com/problems/count-good-meals/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def countPairs(self, deliciousness: List[int]) -> int:
        ...
```

Đây là phiên bản "Two Sum" nâng cấp: thay vì **một** tổng mục tiêu, ta có **22** tổng mục tiêu (các lũy thừa của 2 từ `2⁰` tới `2²¹`). Notebook đi từ duyệt mọi cặp `O(n²)` → sắp xếp + tìm nhị phân → **hash map một lượt** `O(22·n)` → đếm theo **giá trị phân biệt**.

## 1. Mục tiêu học

1. Nhận ra bài **đếm cặp có tổng thuộc một tập nhỏ** và áp dụng mẫu **"đếm phần bù bằng hash map"** (mở rộng từ Two Sum).
2. Kiểm tra một số có phải **lũy thừa của 2** bằng thao tác bit `x > 0 and x & (x - 1) == 0`, và hiểu vì sao phải loại `0`.
3. Xác định chính xác **phạm vi lũy thừa cần thử** từ ràng buộc (`0 ≤ dᵢ ≤ 2²⁰` ⇒ tổng ≤ `2²¹`) — không thiếu, không thừa quá nhiều.
4. Đếm cặp **chỉ số khác nhau** mà không đếm trùng: kỹ thuật "chỉ ghép với phần tử đứng trước" hoặc "đếm theo giá trị rồi xử lý riêng trường hợp `a = b`".
5. Kiểm thử bằng brute force trên dữ liệu ngẫu nhiên có seed cố định, và thấy tác dụng của `mod 10⁹ + 7`.

## 2. Đề bài & ví dụ

Một **bữa ăn ngon** gồm **đúng hai món khác nhau** có tổng độ ngon là **một lũy thừa của 2** (`1, 2, 4, 8, ...`).

Cho mảng `deliciousness`, `deliciousness[i]` là độ ngon của món thứ `i`. Hãy đếm số bữa ăn ngon khác nhau có thể tạo, lấy **modulo `10⁹ + 7`**.

Hai món có **chỉ số khác nhau** được coi là khác nhau dù cùng độ ngon. Nói cách khác: đếm số cặp chỉ số `(i, j)` với `i < j` sao cho `d[i] + d[j]` là lũy thừa của 2.

**Ví dụ 1:** `deliciousness = [1,3,5,7,9]` → `4`. Các cặp: `(1,3)=4`, `(1,7)=8`, `(3,5)=8`, `(7,9)=16`.

**Ví dụ 2:** `deliciousness = [1,1,1,3,3,3,7]` → `15`. `(1,1)` có 3 cách, `(1,3)` có 9 cách, `(1,7)` có 3 cách.

**Ràng buộc:**
- `1 ≤ deliciousness.length ≤ 10⁵`
- `0 ≤ deliciousness[i] ≤ 2²⁰`

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict
from typing import List

MOD = 10**9 + 7

examples = [
    ([1, 3, 5, 7, 9], 4),
    ([1, 1, 1, 3, 3, 3, 7], 15),
]
for d, want in examples:
    print(d, "-> kỳ vọng", want)

[1, 3, 5, 7, 9] -> kỳ vọng 4
[1, 1, 1, 3, 3, 3, 7] -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1 Lũy thừa của 2 trong hệ nhị phân

Lũy thừa của 2 có **đúng một bit 1**: `1 = 0b1`, `2 = 0b10`, `4 = 0b100`, `8 = 0b1000`. Phép `x & (x - 1)` **xóa bit 1 thấp nhất** của `x`. Vì vậy `x` là lũy thừa của 2 ⇔ `x > 0` và `x & (x - 1) == 0`.

Chú ý `0 & (0 - 1) = 0 & -1 = 0` — nếu quên điều kiện `x > 0`, số `0` bị coi nhầm là lũy thừa của 2.

In [2]:
def is_pow2(x):
    return x > 0 and x & (x - 1) == 0

for x in [0, 1, 2, 3, 4, 6, 8, 12, 16, 1 << 21]:
    print(f"{x:>8} = {bin(x):>24}  x&(x-1)={x & (x - 1):>8}  pow2? {is_pow2(x)}")

       0 =                      0b0  x&(x-1)=       0  pow2? False
       1 =                      0b1  x&(x-1)=       0  pow2? True
       2 =                     0b10  x&(x-1)=       0  pow2? True
       3 =                     0b11  x&(x-1)=       2  pow2? False
       4 =                    0b100  x&(x-1)=       0  pow2? True
       6 =                    0b110  x&(x-1)=       4  pow2? False
       8 =                   0b1000  x&(x-1)=       0  pow2? True
      12 =                   0b1100  x&(x-1)=       8  pow2? False
      16 =                  0b10000  x&(x-1)=       0  pow2? True
 2097152 = 0b1000000000000000000000  x&(x-1)=       0  pow2? True


### 3.2 Dịch bit `1 << k`

`1 << k` bằng `2ᵏ`. Danh sách mọi lũy thừa từ `2⁰` tới `2²¹` là `[1 << k for k in range(22)]` — chỉ **22 số**.

In [3]:
POWERS = [1 << k for k in range(22)]
print(len(POWERS), POWERS[:6], "...", POWERS[-2:])

22 [1, 2, 4, 8, 16, 32] ... [1048576, 2097152]


### 3.3 `Counter` và `defaultdict(int)`

`Counter` (trong `collections`) đếm số lần xuất hiện. Truy cập khóa chưa có trả về `0` **mà không thêm khóa** — rất tiện để hỏi "đã thấy bao nhiêu số bằng `x`?".

In [4]:
cnt = Counter([1, 1, 3, 7, 3, 1])
print(cnt)
print("cnt[1] =", cnt[1], "| cnt[100] =", cnt[100], "| 100 in cnt?", 100 in cnt)
cnt[5] += 1
print(cnt)

Counter({1: 3, 3: 2, 7: 1})
cnt[1] = 3 | cnt[100] = 0 | 100 in cnt? False
Counter({1: 3, 3: 2, 7: 1, 5: 1})


### 3.4 Đếm cặp từ tần suất: `C(m, 2) = m·(m−1)/2`

Nếu một giá trị xuất hiện `m` lần, số cặp chỉ số **cùng giá trị** là `m·(m−1)/2`. Nếu giá trị `a` xuất hiện `p` lần và `b ≠ a` xuất hiện `q` lần, số cặp `(a, b)` là `p·q`.

In [5]:
from math import comb
for m in range(1, 6):
    print(f"m={m}: m*(m-1)//2 = {m * (m - 1) // 2}, comb(m,2) = {comb(m, 2)}")

m=1: m*(m-1)//2 = 0, comb(m,2) = 0
m=2: m*(m-1)//2 = 1, comb(m,2) = 1
m=3: m*(m-1)//2 = 3, comb(m,2) = 3
m=4: m*(m-1)//2 = 6, comb(m,2) = 6
m=5: m*(m-1)//2 = 10, comb(m,2) = 10


### 3.5 `bisect` trên mảng đã sắp xếp

`bisect_left(a, x)` trả về vị trí đầu tiên `≥ x`, `bisect_right(a, x)` trả về vị trí đầu tiên `> x`. Số phần tử bằng `x` trong đoạn `a[lo:]` là `bisect_right(a, x, lo) - bisect_left(a, x, lo)`.

In [6]:
a = [1, 1, 3, 3, 3, 7, 9]
x = 3
print("bisect_left :", bisect_left(a, x), "| bisect_right:", bisect_right(a, x))
print("số lượng 3 trong a[4:]:", bisect_right(a, 3, 4) - bisect_left(a, 3, 4))

bisect_left : 2 | bisect_right: 5
số lượng 3 trong a[4:]: 1


### 3.6 Modulo trong Python

Số nguyên Python không tràn, nhưng đề yêu cầu trả về `ans % (10⁹ + 7)`. Có thể cộng dồn rồi `%` một lần cuối, hoặc `%` sau mỗi lần cộng (giống C++/Java).

## 4. Ý tưởng nền tảng / trực giác

**Tính tay Ví dụ 1** `[1,3,5,7,9]`: có `C(5,2) = 10` cặp:

| Cặp | Tổng | Lũy thừa 2? |
|---|---|---|
| 1+3 | 4 | ✔ |
| 1+5 | 6 | ✘ |
| 1+7 | 8 | ✔ |
| 1+9 | 10 | ✘ |
| 3+5 | 8 | ✔ |
| 3+7 | 10 | ✘ |
| 3+9 | 12 | ✘ |
| 5+7 | 12 | ✘ |
| 5+9 | 14 | ✘ |
| 7+9 | 16 | ✔ |

⇒ 4 cặp.

**Đổi góc nhìn:** thay vì hỏi "tổng của cặp này có phải lũy thừa 2 không?", hãy hỏi: "**với món `x`, cần món kia bằng bao nhiêu?**" — đáp: một trong các số `2ᵏ − x`. Có tối đa 22 giá trị như vậy. Ví dụ với `x = 3`: cần `1−3, 2−3, 4−3=1, 8−3=5, 16−3=13, …` ⇒ trong mảng có `1` và `5` ⇒ hai cặp chứa `3`.

**Tổng lớn nhất:** `2²⁰ + 2²⁰ = 2²¹`. Vậy chỉ cần thử `2⁰, 2¹, …, 2²¹` — **22 lũy thừa**.

In [7]:
d = [1, 3, 5, 7, 9]
for x in d:
    need = [p - x for p in POWERS[:6] if p - x >= 0]
    hit = [y for y in need if y in d and y != x]
    print(f"x={x}: các phần bù >= 0 (tới 32): {need} -> có trong mảng: {hit}")

x=1: các phần bù >= 0 (tới 32): [0, 1, 3, 7, 15, 31] -> có trong mảng: [3, 7]
x=3: các phần bù >= 0 (tới 32): [1, 5, 13, 29] -> có trong mảng: [1, 5]
x=5: các phần bù >= 0 (tới 32): [3, 11, 27] -> có trong mảng: [3]
x=7: các phần bù >= 0 (tới 32): [1, 9, 25] -> có trong mảng: [1, 9]
x=9: các phần bù >= 0 (tới 32): [7, 23] -> có trong mảng: [7]


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp

**Ý tưởng:** hai vòng lặp `i < j`, kiểm tra `is_pow2(d[i] + d[j])`.

In [8]:
def count_brute(d):
    n = len(d)
    ans = 0
    for i in range(n):
        for j in range(i + 1, n):
            if is_pow2(d[i] + d[j]):
                ans += 1
    return ans % MOD

for d, want in examples:
    print(count_brute(d), "| kỳ vọng", want)

4 | kỳ vọng 4
15 | kỳ vọng 15


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

**Nút thắt:** với `n = 10⁵` có khoảng `5·10⁹` cặp — quá chậm (Python làm ~10⁷ phép/giây). Ta đo nhanh để thấy tốc độ tăng theo bình phương:

In [9]:
rng = random.Random(0)
for n in [500, 1000, 2000]:
    d = [rng.randint(0, 1 << 20) for _ in range(n)]
    t = time.perf_counter()
    count_brute(d)
    dt = time.perf_counter() - t
    print(f"n={n:>5}: {dt:.3f}s  -> ước lượng n=1e5: ~{dt * (1e5 / n) ** 2 / 60:.0f} phút")

n=  500: 0.014s  -> ước lượng n=1e5: ~9 phút
n= 1000: 0.053s  -> ước lượng n=1e5: ~9 phút


n= 2000: 0.164s  -> ước lượng n=1e5: ~7 phút


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** số tổng mục tiêu rất **ít** (22), còn số phần tử rất **nhiều** (10⁵). Hãy lặp qua **phần tử × mục tiêu** (`22·n ≈ 2,2·10⁶`) thay vì **phần tử × phần tử** (`n²/2 ≈ 5·10⁹`).

Với mỗi phần tử `x = d[j]` và mỗi lũy thừa `p`, số cặp `(i, j)` với `i < j` và `d[i] = p − x` chính là **số lần giá trị `p − x` xuất hiện trong phần đã duyệt** `d[0..j−1]`. Hash map trả lời câu hỏi này trong `O(1)`.

> **Quy tắc:** duyệt `j` từ trái sang phải. *Trước khi* thêm `d[j]` vào bộ đếm, cộng `cnt[p − d[j]]` với mọi `p`. *Sau đó* mới `cnt[d[j]] += 1`.

**Invariant:** ngay trước khi xử lý `j`, `cnt` chứa tần suất của **đúng** `d[0..j−1]`, và `ans` = số cặp tốt `(i, i')` với `i < i' < j`.

Vì sao "trước rồi mới thêm"? Để không ghép `d[j]` với **chính nó** (ví dụ `d[j] = 1`, `p = 2`: nếu thêm trước thì `cnt[1]` đã tính cả `d[j]`).

**Mỗi cặp chỉ được đếm một lần:** cặp `(i, j)` với `i < j` có tổng là **đúng một** lũy thừa `p` (một số không thể bằng hai lũy thừa khác nhau), và nó được đếm đúng một lần — khi duyệt tới `j`, với lũy thừa `p` đó.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + tìm nhị phân cho mỗi lũy thừa

**Từng đoạn:**
1. Sắp xếp `a = sorted(d)`. Thứ tự chỉ số không quan trọng với bài đếm cặp (chỉ cần đếm cặp vị trí khác nhau).
2. Với mỗi vị trí `i` và mỗi lũy thừa `p`, giá trị cần là `y = p − a[i]`. Chỉ đếm `y` trong đoạn `a[i+1:]` (để mỗi cặp vị trí đếm một lần) bằng hai lần `bisect`.
3. Mẹo cắt tỉa: vì `a` tăng dần, cần `y ≥ a[i]`, tức chỉ thử `p ≥ 2·a[i]`.

Đây là bước trung gian "không cần hash map", hay dùng khi chưa quen `Counter`.

In [10]:
def count_sort_bisect(d):
    a = sorted(d)
    n = len(a)
    ans = 0
    for i in range(n):
        x = a[i]
        for p in POWERS:
            y = p - x
            if y < x:                     # phần tử đứng sau i luôn >= x
                continue
            ans += bisect_right(a, y, i + 1) - bisect_left(a, y, i + 1)
    return ans % MOD

for d, want in examples:
    print(count_sort_bisect(d), "| kỳ vọng", want)

4 | kỳ vọng 4
15 | kỳ vọng 15


Code nộp LeetCode của Cách 2:

In [11]:
class SolutionBisect:
    def countPairs(self, deliciousness: List[int]) -> int:
        a = sorted(deliciousness)
        n = len(a)
        powers = [1 << k for k in range(22)]
        ans = 0
        for i in range(n):
            x = a[i]
            for p in powers:
                y = p - x
                if y < x:
                    continue
                ans += bisect_right(a, y, i + 1) - bisect_left(a, y, i + 1)
        return ans % (10**9 + 7)

print(SolutionBisect().countPairs(examples[1][0]))

15


**Độ phức tạp Cách 2:** sắp xếp `O(n log n)` + `22·n` lần bisect `O(log n)` ⇒ Time `O(22·n·log n)` ≈ `3,7·10⁷` phép cơ bản, Space `O(n)` cho mảng đã sắp. Chạy được nhưng chậm hơn hash map.

### 7.2 Cách 3 — Hash map một lượt (đếm phần bù)

**Từng đoạn:**
1. `cnt = Counter()` rỗng, `ans = 0`.
2. Với mỗi `x` trong mảng (theo thứ tự): với mỗi `p` trong 22 lũy thừa, `ans += cnt[p − x]` (nếu `p − x < 0` thì `cnt` trả `0`, không sao; có thể `continue` sớm cho nhanh).
3. `cnt[x] += 1`.
4. Trả `ans % MOD`.

Tối ưu nhỏ: chỉ cần thử `p ≤ 2·max(d)`; và nên dùng `dict.get` thay vì `Counter[...]` để tránh tạo khóa (với `Counter` thì không tạo khóa, nhưng với `defaultdict` thì **có**!).

In [12]:
def count_hash_one_pass(d):
    cnt = {}
    ans = 0
    limit = 2 * max(d)
    pw = [p for p in POWERS if p <= max(limit, 1)]
    for x in d:
        for p in pw:
            y = p - x
            if y >= 0:
                ans += cnt.get(y, 0)
        cnt[x] = cnt.get(x, 0) + 1
    return ans % MOD

for d, want in examples:
    print(count_hash_one_pass(d), "| kỳ vọng", want)

4 | kỳ vọng 4
15 | kỳ vọng 15


Code nộp LeetCode của Cách 3 (bản nộp chính, xem lại ở mục 9):

In [13]:
class Solution:
    def countPairs(self, deliciousness: List[int]) -> int:
        MOD = 10**9 + 7
        mx2 = 2 * max(deliciousness)
        powers = [1 << k for k in range(22) if (1 << k) <= max(mx2, 1)]
        cnt = {}
        ans = 0
        for x in deliciousness:
            for p in powers:
                y = p - x
                if y >= 0:
                    ans += cnt.get(y, 0)
            cnt[x] = cnt.get(x, 0) + 1
        return ans % MOD

print([Solution().countPairs(d) for d, _ in examples])

[4, 15]


**Độ phức tạp Cách 3:** Time `O(22·n)` (trung bình, nhờ hash map `O(1)`), Space `O(n)` cho `cnt`.

### 7.3 Cách 4 — Đếm theo **giá trị phân biệt**

Nếu mảng có nhiều giá trị lặp lại (ví dụ `10⁵` phần tử nhưng chỉ vài trăm giá trị khác nhau), ta làm việc trên `Counter(d)` gồm `k` giá trị phân biệt:

**Từng đoạn:**
1. `cnt = Counter(d)`.
2. Với mỗi giá trị `a` và mỗi lũy thừa `p`: `b = p − a`. Chỉ xét `b ≥ a` để mỗi cặp **giá trị** được tính một lần.
3. Nếu `b == a`: cộng `C(m, 2) = m·(m−1)/2` với `m = cnt[a]` (chọn 2 trong `m` món cùng giá trị).
4. Nếu `b > a` và `b` có trong `cnt`: cộng `cnt[a]·cnt[b]`.

Kiểm tra bằng Ví dụ 2: `cnt = {1:3, 3:3, 7:1}`. `(1,1)` → `C(3,2)=3`; `(1,3)` → `3·3 = 9`; `(1,7)` → `3·1 = 3`. Tổng `15`. ✔

In [14]:
def count_by_values(d):
    cnt = Counter(d)
    ans = 0
    for a, ma in cnt.items():
        for p in POWERS:
            b = p - a
            if b < a:
                continue
            if b == a:
                ans += ma * (ma - 1) // 2
            elif b in cnt:
                ans += ma * cnt[b]
    return ans % MOD

for d, want in examples:
    print(count_by_values(d), "| kỳ vọng", want)

4 | kỳ vọng 4
15 | kỳ vọng 15


Code nộp LeetCode của Cách 4:

In [15]:
class SolutionValues:
    def countPairs(self, deliciousness: List[int]) -> int:
        cnt = Counter(deliciousness)
        ans = 0
        for a, ma in cnt.items():
            for k in range(22):
                b = (1 << k) - a
                if b < a:
                    continue
                if b == a:
                    ans += ma * (ma - 1) // 2
                elif b in cnt:
                    ans += ma * cnt[b]
        return ans % (10**9 + 7)

print([SolutionValues().countPairs(d) for d, _ in examples])

[4, 15]


**Độ phức tạp Cách 4:** Time `O(n + 22·k)` với `k` = số giá trị phân biệt (`k ≤ n`), Space `O(k)`. Nhanh nhất khi dữ liệu nhiều trùng lặp.

## 8. TRACE từng bước

Trace **Cách 3** trên Ví dụ 2 `[1,1,1,3,3,3,7]`. Bảng trạng thái (chỉ ghi các lũy thừa có đóng góp):

| `j` | `x` | `cnt` trước | đóng góp | `ans` sau |
|---|---|---|---|---|
| 0 | 1 | `{}` | — | 0 |
| 1 | 1 | `{1:1}` | `p=2`: cnt[1]=1 | 1 |
| 2 | 1 | `{1:2}` | `p=2`: cnt[1]=2 | 3 |
| 3 | 3 | `{1:3}` | `p=4`: cnt[1]=3 | 6 |
| 4 | 3 | `{1:3,3:1}` | `p=4`: cnt[1]=3 | 9 |
| 5 | 3 | `{1:3,3:2}` | `p=4`: cnt[1]=3 | 12 |
| 6 | 7 | `{1:3,3:3}` | `p=8`: cnt[1]=3 | 15 |

Chú ý ở `j = 4`: `p = 8` cần `5`, `p = 2` cần `−1`, `p = 16` cần `13` — đều không có. Và `3 + 3 = 6` không phải lũy thừa 2 nên các món `3` không ghép với nhau.

In [16]:
def trace_hash(d):
    cnt = {}
    ans = 0
    pw = [p for p in POWERS if p <= max(2 * max(d), 1)]
    for j, x in enumerate(d):
        before = dict(cnt)
        contrib = []
        for p in pw:
            y = p - x
            if y >= 0 and cnt.get(y, 0):
                contrib.append(f"p={p}:cnt[{y}]={cnt[y]}")
                ans += cnt[y]
        cnt[x] = cnt.get(x, 0) + 1
        print(f"j={j} x={x} cnt_trước={before} đóng_góp={contrib or '—'} ans={ans}")
    return ans

print("Kết quả:", trace_hash([1, 1, 1, 3, 3, 3, 7]))

j=0 x=1 cnt_trước={} đóng_góp=— ans=0
j=1 x=1 cnt_trước={1: 1} đóng_góp=['p=2:cnt[1]=1'] ans=1
j=2 x=1 cnt_trước={1: 2} đóng_góp=['p=2:cnt[1]=2'] ans=3
j=3 x=3 cnt_trước={1: 3} đóng_góp=['p=4:cnt[1]=3'] ans=6
j=4 x=3 cnt_trước={1: 3, 3: 1} đóng_góp=['p=4:cnt[1]=3'] ans=9
j=5 x=3 cnt_trước={1: 3, 3: 2} đóng_góp=['p=4:cnt[1]=3'] ans=12
j=6 x=7 cnt_trước={1: 3, 3: 3} đóng_góp=['p=8:cnt[1]=3'] ans=15
Kết quả: 15


Trace **Cách 4** (theo giá trị) trên cùng ví dụ:

In [17]:
def trace_values(d):
    cnt = Counter(d)
    print("cnt =", dict(cnt))
    ans = 0
    for a, ma in sorted(cnt.items()):
        for p in POWERS:
            b = p - a
            if b < a:
                continue
            if b == a:
                add = ma * (ma - 1) // 2
                if add:
                    print(f"  a={a}, p={p}: b=a, C({ma},2) = {add}")
                ans += add
            elif b in cnt:
                add = ma * cnt[b]
                print(f"  a={a}, p={p}: b={b}, {ma}*{cnt[b]} = {add}")
                ans += add
    print("tổng =", ans)
    return ans

trace_values([1, 1, 1, 3, 3, 3, 7])

cnt = {1: 3, 3: 3, 7: 1}
  a=1, p=2: b=a, C(3,2) = 3
  a=1, p=4: b=3, 3*3 = 9
  a=1, p=8: b=7, 3*1 = 3
tổng = 15


15

## 9. Code nộp LeetCode

Bản nộp chính: hash map một lượt (Cách 3).

In [18]:
class Solution:
    def countPairs(self, deliciousness: List[int]) -> int:
        MOD = 10**9 + 7
        mx2 = 2 * max(deliciousness)                 # tổng lớn nhất có thể
        powers = [1 << k for k in range(22) if (1 << k) <= max(mx2, 1)]
        cnt = {}                                    # tần suất các món đã duyệt
        ans = 0
        for x in deliciousness:
            for p in powers:
                y = p - x                           # món cần ghép
                if y >= 0:
                    ans += cnt.get(y, 0)
            cnt[x] = cnt.get(x, 0) + 1              # thêm SAU khi đếm
        return ans % MOD

sol = Solution()
for d, want in examples:
    assert sol.countPairs(d) == want
print("Solution qua 2 ví dụ của đề")

Solution qua 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Song ánh cặp ↔ lần cộng.** Gọi `G` là tập cặp chỉ số `(i, j)` với `i < j` và `d[i] + d[j]` là lũy thừa 2. Mỗi lần thuật toán cộng `cnt[p − x]` tại bước `j` (với `x = d[j]`) chính là cộng số chỉ số `i < j` có `d[i] = p − d[j]`, tức số cặp `(i, j) ∈ G` có tổng bằng **đúng `p`**.

- Mọi cặp `(i, j) ∈ G` có tổng bằng một lũy thừa duy nhất `p*` (vì các lũy thừa 2 phân biệt), và `p* ≤ 2²¹` (do `d ≤ 2²⁰`), nên `p*` nằm trong danh sách được thử. Cặp này được đếm khi xét `j` với `p = p*` — **đúng một lần**.
- Không cặp nào ngoài `G` được đếm, và không có cặp `(j, j)` vì `cnt` chưa chứa `d[j]` lúc đếm.

**Cắt tỉa `p ≤ 2·max(d)` đúng** vì mọi tổng ≤ `2·max(d)`. Trường hợp `max(d) = 0` (toàn số 0): tổng luôn là `0`, không phải lũy thừa 2 ⇒ đáp án 0; ta giữ `p = 1` trong danh sách (do `max(mx2, 1)`) cho an toàn, kết quả vẫn 0 vì cần `y = 1` không có.

**Cách 4:** mỗi cặp **giá trị** không thứ tự `{a, b}` (`a ≤ b`) với `a + b = p` được xét đúng một lần (khi duyệt `a`, lũy thừa `p`); số cặp chỉ số tương ứng là `cnt[a]·cnt[b]` nếu `a ≠ b`, `C(cnt[a], 2)` nếu `a = b`.

## 11. Phân tích Time / Space Complexity

Gọi `n` = độ dài mảng (`≤ 10⁵`), `L` = số lũy thừa cần thử (`≤ 22`), `k` = số giá trị phân biệt.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi cặp | `O(n²)` | `O(1)` | ~5·10⁹ phép khi `n = 10⁵` ⇒ TLE |
| 2. Sắp xếp + bisect | `O(n log n + L·n·log n)` | `O(n)` | chạy được, hằng số lớn |
| 3. Hash một lượt | `O(L·n)` trung bình | `O(n)` | **nộp bài** |
| 4. Theo giá trị | `O(n + L·k)` trung bình | `O(k)` | nhanh nhất khi nhiều trùng |

Hash map có **worst case** `O(n)` mỗi thao tác về lý thuyết (va chạm), nhưng với số nguyên trong Python thực tế là `O(1)`.

Đo trên `n = 10⁵` (kích thước tối đa):

In [19]:
rng = random.Random(1711)
big = [rng.randint(0, 1 << 20) for _ in range(10**5)]
big_dup = [rng.choice([1, 3, 5, 7, 1 << 19, (1 << 19) + 1, 0]) for _ in range(10**5)]
for name, f in [("bisect", count_sort_bisect), ("hash", count_hash_one_pass), ("values", count_by_values)]:
    for label, arr in [("ngẫu nhiên", big), ("nhiều trùng", big_dup)]:
        t = time.perf_counter()
        r = f(arr)
        print(f"{name:>7} | {label:<11}: {time.perf_counter() - t:.3f}s  ans={r}")

 bisect | ngẫu nhiên : 0.131s  ans=9420


 bisect | nhiều trùng: 0.399s  ans=225138209
   hash | ngẫu nhiên : 0.076s  ans=9420
   hash | nhiều trùng: 0.089s  ans=225138209


 values | ngẫu nhiên : 0.072s  ans=9420
 values | nhiều trùng: 0.004s  ans=225138209


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

- `[0]`: một món ⇒ 0.
- `[0, 0]`: tổng 0 **không** phải lũy thừa 2 ⇒ 0.
- `[0, 1]`: tổng 1 = `2⁰` ⇒ 1.
- `[1]`: không ghép với chính nó ⇒ 0.
- `[2²⁰, 2²⁰]`: tổng `2²¹` ⇒ 1 (kiểm tra đủ 22 lũy thừa).
- `[2, 2, 2]`: `C(3, 2) = 3`.
- `[0, 0, 4, 4]`: `(0,4)` × 4 = 4; `(4,4) = 8` ⇒ +1; `(0,0)` không ⇒ 5.

In [20]:
impls = {
    "brute": count_brute,
    "bisect": count_sort_bisect,
    "hash": count_hash_one_pass,
    "values": count_by_values,
    "Solution": Solution().countPairs,
    "SolutionBisect": SolutionBisect().countPairs,
    "SolutionValues": SolutionValues().countPairs,
}
edge = [
    ([0], 0), ([0, 0], 0), ([0, 1], 1), ([1], 0),
    ([1 << 20, 1 << 20], 1), ([2, 2, 2], 3), ([0, 0, 4, 4], 5),
    ([1 << 20, (1 << 20) - 1, 1], 1),
]
for d, want in examples + edge:
    for name, f in impls.items():
        assert f(d) == want, (name, d, f(d), want)
print("Tất cả cách qua", len(examples) + len(edge), "test cố định")

Tất cả cách qua 10 test cố định


### 12.2 Ngẫu nhiên có seed, so sánh với brute force

Hai kiểu sinh dữ liệu: (a) giá trị nhỏ `0..40` để có nhiều tổng là lũy thừa 2 và nhiều trùng lặp; (b) giá trị trên toàn miền `0..2²⁰`, trộn thêm lũy thừa 2 và phần bù để chắc có cặp tốt.

In [21]:
rng = random.Random(2021)
tests = 0
for _ in range(400):
    n = rng.randint(1, 60)
    d = [rng.randint(0, 40) for _ in range(n)]
    ref = count_brute(d)
    for name, f in impls.items():
        assert f(d) == ref, (name, d)
    tests += 1
for _ in range(150):
    n = rng.randint(1, 120)
    d = []
    for _ in range(n):
        r = rng.random()
        if r < 0.3:
            d.append(1 << rng.randint(0, 20))
        elif r < 0.6 and d:
            p = 1 << rng.randint(0, 21)
            y = p - rng.choice(d)
            d.append(y if 0 <= y <= (1 << 20) else rng.randint(0, 1 << 20))
        else:
            d.append(rng.randint(0, 1 << 20))
    ref = count_brute(d)
    for name, f in impls.items():
        assert f(d) == ref, (name, d)
    tests += 1
print("OK", tests, "bộ ngẫu nhiên, mọi cách khớp brute force")

OK

 550 bộ ngẫu nhiên, mọi cách khớp brute force


### 12.3 Kiểm tra modulo thật sự có tác dụng

`10⁵` món đều bằng `1`: mọi cặp có tổng `2` ⇒ `C(10⁵, 2) = 4 999 950 000 > 10⁹ + 7`, nên đáp án phải là số đã lấy modulo.

In [22]:
ones = [1] * 10**5
raw = 10**5 * (10**5 - 1) // 2
print("số cặp thật:", raw, "| sau mod:", raw % MOD)
for name in ["hash", "values", "Solution"]:
    assert impls[name](ones) == raw % MOD
print("Các cách hash/values đều trả về đáp án đã mod")

số cặp thật: 4999950000 | sau mod: 999949972
Các cách hash/values đều trả về đáp án đã mod


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi `0` là lũy thừa của 2

In [23]:
def wrong_zero(d):
    n = len(d)
    ans = 0
    for i in range(n):
        for j in range(i + 1, n):
            s = d[i] + d[j]
            if s & (s - 1) == 0:            # SAI: thiếu s > 0
                ans += 1
    return ans

print("sai :", wrong_zero([0, 0, 0]), "| đúng:", count_brute([0, 0, 0]))

sai : 3 | đúng: 0


**Vì sao sai:** `0 & (−1) = 0` nên điều kiện bit đúng cả với `0`. Ràng buộc cho phép `d[i] = 0`, nên hai món `0` có tổng `0` bị đếm nhầm. **Sửa:** `s > 0 and s & (s - 1) == 0`. (Ở cách hash, lỗi tương tự là đưa `p = 0` vào danh sách lũy thừa.)

### Lỗi 2 — Thiếu lũy thừa lớn nhất `2²¹`

In [24]:
def wrong_range(d):
    cnt = {}
    ans = 0
    for x in d:
        for k in range(21):                 # SAI: chỉ tới 2^20
            ans += cnt.get((1 << k) - x, 0)
        cnt[x] = cnt.get(x, 0) + 1
    return ans % MOD

d = [1 << 20, 1 << 20]
print("sai :", wrong_range(d), "| đúng:", Solution().countPairs(d))

sai : 0 | đúng: 1


**Vì sao sai:** mỗi món tối đa `2²⁰`, nên tổng tối đa `2²¹` — phải thử **22** lũy thừa `2⁰..2²¹`. Nhiều người lấy "≤ 2²⁰" từ đề rồi quên rằng đó là giới hạn **mỗi món**, không phải **tổng**. **Sửa:** `range(22)` hoặc lặp tới `2·max(d)`.

### Lỗi 3 — Thêm vào bộ đếm **trước** khi tra

In [25]:
def wrong_add_first(d):
    cnt = {}
    ans = 0
    for x in d:
        cnt[x] = cnt.get(x, 0) + 1          # SAI: thêm trước
        for k in range(22):
            ans += cnt.get((1 << k) - x, 0)
    return ans % MOD

for d in ([1], [2, 2], [1, 3]):
    print(d, "sai:", wrong_add_first(d), "| đúng:", Solution().countPairs(d))

[1] sai: 1 | đúng: 0
[2, 2] sai: 3 | đúng: 1
[1, 3] sai: 2 | đúng: 1


**Vì sao sai:** khi `x` là "nửa" của một lũy thừa (`1 = 2/2`, `2 = 4/2`, …), `cnt[p − x] = cnt[x]` đã chứa chính `x`, nên món bị ghép **với chính nó**. `[1]` cho ra 1 thay vì 0. **Sửa:** tra trước, thêm sau.

### Lỗi 4 — Đếm theo giá trị nhưng tính mỗi cặp hai lần

In [26]:
def wrong_double(d):
    cnt = Counter(d)
    ans = 0
    for a, ma in cnt.items():
        for k in range(22):
            b = (1 << k) - a
            if b == a:
                ans += ma * (ma - 1) // 2
            elif b in cnt:                  # SAI: không ép b > a
                ans += ma * cnt[b]
    return ans % MOD

d = [1, 3, 5, 7, 9]
print("sai :", wrong_double(d), "| đúng:", Solution().countPairs(d))

sai : 8 | đúng: 4


**Vì sao sai:** cặp giá trị `{1, 3}` được cộng một lần khi `a = 1, b = 3` và một lần nữa khi `a = 3, b = 1`. Kết quả gấp đôi phần `a ≠ b`. **Sửa:** chỉ xét `b ≥ a` (hoặc cộng hết rồi chia 2 phần `a ≠ b` — cẩn thận phần `a = b`).

### Lỗi 5 — Dùng `defaultdict` để tra cứu làm phình bộ nhớ và chậm

In [27]:
def wrong_defaultdict(d):
    cnt = defaultdict(int)
    ans = 0
    for x in d:
        for k in range(22):
            ans += cnt[(1 << k) - x]        # mỗi lần tra tạo khóa mới
        cnt[x] += 1
    return ans % MOD, len(cnt)

rng = random.Random(3)
d = [rng.randint(0, 1 << 20) for _ in range(3000)]
ans_w, size_w = wrong_defaultdict(d)
print("kết quả vẫn đúng?", ans_w == Solution().countPairs(d), "| số khóa trong dict:", size_w, "(chỉ có", len(set(d)), "giá trị thật)")

kết quả vẫn đúng? True | số khóa trong dict: 67350 (chỉ có 2994 giá trị thật)


**Vì sao "sai":** kết quả vẫn đúng, nhưng `defaultdict[key]` **tạo khóa** mỗi khi tra (kể cả khóa âm như `1 − x`). Với `n = 10⁵` ta có thể tạo tới `2,2·10⁶` khóa rác ⇒ tốn bộ nhớ, chậm, có thể Memory/Time Limit. **Sửa:** dùng `cnt.get(y, 0)` hoặc `Counter` (tra khóa thiếu không thêm khóa), và bỏ qua `y < 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm cặp có tổng = T"** ⇒ hash map đếm phần bù (Two Sum đếm). **"Tổng thuộc tập nhỏ S"** ⇒ lặp `|S|` mục tiêu, Time `O(|S|·n)`.
- **"Lũy thừa của 2"** ⇒ nghĩ ngay tới `1 << k`, `x & (x − 1)`, và **đếm số lũy thừa cần thử** từ ràng buộc (`log₂(2·max) + 1`).
- **"Chỉ số khác nhau, giá trị có thể trùng"** ⇒ duyệt một lượt, *tra trước – thêm sau*; hoặc theo giá trị với công thức `C(m, 2)` cho cặp cùng giá trị.
- **Đọc kỹ ràng buộc biên**: `0` có được phép không? tổng tối đa bao nhiêu? có cần mod không?
- Khi nghi ngờ, viết brute `O(n²)` để đối chiếu trên mảng nhỏ — chỉ mất 5 dòng.

## 15. Biến thể của bài

1. **LC 1 — Two Sum**, **LC 1679 — Max Number of K-Sum Pairs**: một tổng mục tiêu.
2. **LC 1010 — Pairs of Songs With Total Durations Divisible by 60**: tổng chia hết ⇒ đếm theo phần dư.
3. **LC 2183 — Count Array Pairs Divisible by K**: tích chia hết ⇒ đếm theo ước chung lớn nhất.
4. **Tổng là lũy thừa của 3 / số chính phương / số Fibonacci**: đổi tập mục tiêu (bài tập 5).
5. **Đếm bộ ba** có tổng là lũy thừa 2: cố định một phần tử + đếm cặp ⇒ `O(L·n²)`.
6. **Hiệu** (thay vì tổng) là lũy thừa 2: cần `x − y = p` ⇒ tra `x − p` **và** `x + p`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | kiểm tra trực tiếp | `O(n²)` | `O(1)` | baseline, test nhỏ |
| 2. Sort + bisect | đếm phần bù trong mảng đã sắp | `O(L·n log n)` | `O(n)` | không muốn dùng hash, cần thêm thông tin thứ tự |
| 3. Hash một lượt | tra phần bù trước, thêm sau | `O(L·n)` | `O(n)` | **nộp bài** — tổng quát, dễ đúng |
| 4. Theo giá trị | `Counter` + `C(m,2)` / `p·q` | `O(n + L·k)` | `O(k)` | nhiều giá trị lặp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[0, 0, 1]`; (b) `[2, 2, 2, 2]`; (c) `[1, 1, 2, 2]`.

**Bài 2 (tính tay):** Với `[1, 7, 9, 15, 17]`, liệt kê mọi cặp tốt và tổng của chúng, rồi dùng Cách 4 (bảng `cnt`) để tính lại.

**Bài 3 (điền TODO):** Viết `is_power_of_two(x)` **không dùng** toán tử bit (chỉ dùng chia 2 lặp lại).

**Bài 4 (tự viết lại):** Viết lại Cách 3 nhưng trả về thêm **danh sách các cặp chỉ số** `(i, j)` tốt (chỉ dùng cho mảng nhỏ để debug).

**Bài 5 (mini challenge):** Viết `count_pairs_targets(nums, targets)` đếm cặp `i < j` có `nums[i] + nums[j] ∈ targets` (tập bất kỳ), rồi dùng nó đếm cặp có tổng là **lũy thừa của 3**. Kiểm tra với brute force.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_power_of_two_todo(x):
    # TODO: x <= 0 thì False
    # TODO: chia x cho 2 khi x chẵn; cuối cùng x == 1?
    pass

In [29]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def good_pairs_todo(d):
    pos = {}          # giá trị -> danh sách chỉ số đã thấy
    pairs = []
    # TODO: với mỗi j, mỗi lũy thừa p: ghép j với mọi i trong pos[p - d[j]]
    return pairs

In [30]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def count_pairs_targets_todo(nums, targets):
    # TODO: hash một lượt, lặp qua targets thay vì POWERS
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: liệt kê tất cả cặp. Mức 2: `0 + 0 = 0` không tính. Mức 3: (c) có `1+1=2`, `2+2=4`, còn `1+2=3` không.

**Bài 2.** Mức 1: các tổng có thể là 8, 16, 24, 32. Mức 2: `1+7`, `1+15`, `7+9`, `15+17`, `9+7`… cẩn thận không lặp. Mức 3: `cnt` mỗi giá trị là 1 ⇒ mỗi cặp giá trị đóng góp 1.

**Bài 3.** Mức 1: lũy thừa 2 chia 2 liên tục sẽ về 1. Mức 2: nếu gặp số lẻ > 1 thì không phải. Mức 3: nhớ trường hợp `x = 0` (vòng lặp vô hạn nếu không chặn).

**Bài 4.** Mức 1: thay `cnt` bằng `pos: dict[int, list[int]]`. Mức 2: thêm `j` vào `pos[d[j]]` **sau** khi ghép. Mức 3: số cặp tìm được phải bằng `Solution().countPairs(d)` khi chưa mod.

**Bài 5.** Mức 1: lũy thừa của 3 tới `2²¹`: `1, 3, 9, …, 3¹³`. Mức 2: dùng `set` hoặc list cho `targets`. Mức 3: brute dùng `in targets_set`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** (a) `2` — `(0,1)` hai lần, `(0,0)` không tính. (b) `6` — `C(4,2)`, mọi cặp tổng 4. (c) `2` — `(1,1)` và `(2,2)`.

In [31]:
assert Solution().countPairs([0, 0, 1]) == 2
assert Solution().countPairs([2, 2, 2, 2]) == 6
assert Solution().countPairs([1, 1, 2, 2]) == 2
print("Bài 1 đúng")

Bài 1 đúng


**Bài 2:** các cặp tốt: `1+7=8`, `1+15=16`, `7+9=16`, `15+17=32`. (`9+7` là cùng cặp với `7+9`; `1+9=10`, `9+15=24`, `7+17=24` không.) Đáp án `4`.

In [32]:
d = [1, 7, 9, 15, 17]
assert count_brute(d) == 4 and trace_values(d) == 4

cnt = {1: 1, 7: 1, 9: 1, 15: 1, 17: 1}
  a=1, p=8: b=7, 1*1 = 1
  a=1, p=16: b=15, 1*1 = 1
  a=7, p=16: b=9, 1*1 = 1
  a=15, p=32: b=17, 1*1 = 1
tổng = 4


In [33]:
# Bài 3 — đáp án
def is_power_of_two(x):
    if x <= 0:
        return False
    while x % 2 == 0:
        x //= 2
    return x == 1

for x in range(0, 5000):
    assert is_power_of_two(x) == is_pow2(x)
print("Bài 3 đúng")

Bài 3 đúng


In [34]:
# Bài 4 — đáp án
def good_pairs(d):
    pos = defaultdict(list)
    pairs = []
    for j, x in enumerate(d):
        for p in POWERS:
            y = p - x
            if y >= 0 and y in pos:
                for i in pos[y]:
                    pairs.append((i, j))
        pos[x].append(j)
    return sorted(pairs)

print(good_pairs([1, 3, 5, 7, 9]))
rng = random.Random(4)
for _ in range(100):
    d = [rng.randint(0, 20) for _ in range(rng.randint(1, 25))]
    ps = good_pairs(d)
    assert len(ps) == count_brute(d)
    assert all(i < j and is_pow2(d[i] + d[j]) for i, j in ps)
print("Bài 4 đúng")

[(0, 1), (0, 3), (1, 2), (3, 4)]
Bài 4 đúng


In [35]:
# Bài 5 — đáp án
def count_pairs_targets(nums, targets):
    targets = sorted(set(targets))
    cnt = {}
    ans = 0
    for x in nums:
        for t in targets:
            y = t - x
            if y >= 0:
                ans += cnt.get(y, 0)
        cnt[x] = cnt.get(x, 0) + 1
    return ans

POW3 = [3 ** k for k in range(14)]            # 3^13 = 1594323 < 2^21
def brute_targets(nums, targets):
    s = set(targets)
    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if nums[i] + nums[j] in s)

rng = random.Random(5)
for _ in range(200):
    nums = [rng.randint(0, 100) for _ in range(rng.randint(1, 40))]
    assert count_pairs_targets(nums, POW3) == brute_targets(nums, POW3)
    assert count_pairs_targets(nums, POWERS) % MOD == Solution().countPairs(nums)
print("lũy thừa 3 của [1,2,8,1,26]:", count_pairs_targets([1, 2, 8, 1, 26], POW3))
print("Bài 5 đúng")

lũy thừa 3 của [1,2,8,1,26]: 6
Bài 5 đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Template: đếm cặp i < j có tổng thuộc tập mục tiêu nhỏ
cnt = {}
ans = 0
for x in nums:
    for t in targets:                 # ở bài này: 1 << k, k = 0..21
        y = t - x
        if y >= 0:
            ans += cnt.get(y, 0)      # tra TRƯỚC
    cnt[x] = cnt.get(x, 0) + 1        # thêm SAU
return ans % MOD

# Kiểm tra lũy thừa 2
is_pow2 = lambda x: x > 0 and x & (x - 1) == 0
```

**Checklist trước khi submit:**
- [ ] Danh sách lũy thừa đủ tới `2²¹` (tổng của hai số `2²⁰`).
- [ ] Không có `p = 0`; nhớ `x > 0` trong kiểm tra bit.
- [ ] Tra bộ đếm **trước**, thêm phần tử **sau** (tránh ghép với chính nó).
- [ ] Nếu đếm theo giá trị: chỉ xét `b ≥ a`, dùng `C(m, 2)` khi `a = b`.
- [ ] Trả về `ans % (10⁹ + 7)`.
- [ ] Thử `[0,0]`, `[1]`, `[2²⁰, 2²⁰]`, và 2 ví dụ của đề.